In [54]:
import pandas as pd
import numpy as np


In [26]:
df = pd.read_csv("../data/books_raw.csv")

df.head()

,title,category,price,rating,availability,product_description,upc,number_of_reviews,product_url
0,It's Only the Himalayas,Travel,£45.17,Two,In stock (19 available),"“Wherever you go, whatever you do, just . . . ...",a22124811bfa8350,0,http://books.toscrape.com/catalogue/its-only-t...
1,Libertarianism for Beginners,Politics,£51.33,Two,In stock (19 available),Libertarianism isn't about winning elections; ...,a18a4f574854aced,0,http://books.toscrape.com/catalogue/libertaria...
2,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,£37.59,One,In stock (19 available),"Andrew Barger, award-winning author and engine...",e30f54cea9b38190,0,http://books.toscrape.com/catalogue/mesaerion-...
3,Olio,Poetry,£23.88,One,In stock (19 available),"Part fact, part fiction, Tyehimba Jess's much ...",feb7cc7701ecf901,0,http://books.toscrape.com/catalogue/olio_984/i...
4,Our Band Could Be Your Life: Scenes from the A...,Music,£57.25,Three,In stock (19 available),This is the never-before-told story of the mus...,deda3e61b9514b83,0,http://books.toscrape.com/catalogue/our-band-c...


In [27]:
print(df.shape)

(200, 9)


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   title                200 non-null    object
 1   category             200 non-null    object
 2   price                200 non-null    object
 3   rating               200 non-null    object
 4   availability         200 non-null    object
 5   product_description  199 non-null    object
 6   upc                  200 non-null    object
 7   number_of_reviews    200 non-null    int64 
 8   product_url          200 non-null    object
dtypes: int64(1), object(8)
memory usage: 14.2+ KB


In [29]:
duplicate_upcs = df["upc"].duplicated().sum()
print(duplicate_upcs)

0


In [30]:
df["price"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [31]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

In [32]:
df["stock"] = (
    df["availability"]
    .str.extract(r"(\d+)")
    .fillna(0)
    .astype(int)
)

In [33]:
df["description_word_count"] = (
    df["product_description"]
    .str.split()
    .str.len()
)

In [34]:
df["price_band"] = pd.qcut(
    df["price"],
    q=4,
    labels=[
        "Cheap",
        "Affordable",
        "Expensive",
        "Luxury"
    ]
)

In [35]:
df["value_score"] = (
    df["rating"] / df["price"]
).round(3)

In [36]:
min_price = df["price"].min()
max_price = df["price"].max()

df["affordability_score"] = (
    10
    - 9 * (df["price"] - min_price) / (max_price - min_price)
).round(1)

In [52]:
df.head()

,title,category,price,rating,availability,product_description,upc,number_of_reviews,product_url,stock,description_word_count,price_band,value_score,affordability_score
0,It's Only the Himalayas,Travel,45.17,2,Available,"“Wherever you go, whatever you do, just . . . ...",a22124811bfa8350,0,http://books.toscrape.com/catalogue/its-only-t...,19,230,Expensive,0.044,3.6
1,Libertarianism for Beginners,Politics,51.33,2,Available,Libertarianism isn't about winning elections; ...,a18a4f574854aced,0,http://books.toscrape.com/catalogue/libertaria...,19,195,Luxury,0.039,2.5
2,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,37.59,1,Available,"Andrew Barger, award-winning author and engine...",e30f54cea9b38190,0,http://books.toscrape.com/catalogue/mesaerion-...,19,283,Expensive,0.027,5.0
3,Olio,Poetry,23.88,1,Available,"Part fact, part fiction, Tyehimba Jess's much ...",feb7cc7701ecf901,0,http://books.toscrape.com/catalogue/olio_984/i...,19,249,Affordable,0.042,7.5
4,Our Band Could Be Your Life: Scenes from the A...,Music,57.25,3,Available,This is the never-before-told story of the mus...,deda3e61b9514b83,0,http://books.toscrape.com/catalogue/our-band-c...,19,163,Luxury,0.052,1.4


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   title                   200 non-null    object  
 1   category                200 non-null    object  
 2   price                   200 non-null    float64 
 3   rating                  200 non-null    int64   
 4   availability            200 non-null    object  
 5   product_description     199 non-null    object  
 6   upc                     200 non-null    object  
 7   number_of_reviews       200 non-null    int64   
 8   product_url             200 non-null    object  
 9   stock                   200 non-null    int64   
 10  description_word_count  199 non-null    float64 
 11  price_band              200 non-null    category
 12  value_score             200 non-null    float64 
 13  affordability_score     200 non-null    float64 
dtypes: category(1), float64(4)

In [49]:
df.describe()

,price,rating,number_of_reviews,stock,description_word_count,value_score,affordability_score
count,200.000000,200.000000,200.0,200.000000,200.000000,200.000000,200.000000
mean,34.796250,2.865000,0.0,16.230000,235.940000,0.101165,5.517000
std,14.119272,1.451658,0.0,1.448305,100.169336,0.076509,2.569753
min,10.160000,1.000000,0.0,15.000000,3.000000,0.017000,1.000000
25%,21.990000,2.000000,0.0,15.000000,176.000000,0.050500,3.475000
50%,35.640000,3.000000,0.0,16.000000,224.000000,0.085000,5.400000
75%,46.110000,4.000000,0.0,16.000000,280.000000,0.130750,7.825000
max,59.640000,5.000000,0.0,22.000000,722.000000,0.397000,10.000000


In [40]:
df["product_description"] = df["product_description"].replace(
    "",
    "No description available"
)

In [41]:
print(df["upc"].duplicated().sum())

0


In [42]:
df["number_of_reviews"] = df["number_of_reviews"].astype(int)

In [43]:
df["product_description"] = df["product_description"].fillna("No description available")

In [44]:
df["description_word_count"] = (
    df["product_description"]
      .str.split()
      .str.len()
      .astype(int)
)

In [46]:
df.isnull().sum()

title                     0
category                  0
price                     0
rating                    0
availability              0
product_description       0
upc                       0
number_of_reviews         0
product_url               0
stock                     0
description_word_count    0
price_band                0
value_score               0
affordability_score       0
dtype: int64

In [53]:
df.to_csv("../data/books_clean.csv", index=False)

In [51]:
df["availability"] = np.where(
    df["stock"] > 0,
    "Available",
    "Not available"
)